# Pythonic FastHTML SSE Integration Example

> Demonstrates a fully Pythonic approach to real-time tqdm progress tracking in FastHTML using Server-Sent Events (SSE). All JavaScript is generated from Python parameters using dataclasses and type hints for better IDE support and maintainability.

In [1]:
from fasthtml.common import *
import uuid, time
from dataclasses import dataclass, field
from typing import Optional, Dict, Any, List
from enum import Enum

from cjm_tqdm_capture.progress_monitor import ProgressMonitor
from cjm_tqdm_capture.job_runner import JobRunner
from cjm_tqdm_capture.streaming import sse_stream

In [2]:
# For Jupyter display
from fasthtml.jupyter import JupyUvi, HTMX
from cjm_fasthtml_daisyui.core.testing import create_test_app, create_test_page, start_test_server
from cjm_fasthtml_daisyui.core.themes import DaisyUITheme
from IPython.display import display

# Import DaisyUI factories
from cjm_fasthtml_daisyui.components.actions.button import btn, btn_colors, btn_behaviors
from cjm_fasthtml_daisyui.components.feedback.progress import progress, progress_colors

# Import Tailwind factories
from cjm_fasthtml_tailwind.utilities.spacing import p, m
from cjm_fasthtml_tailwind.utilities.typography import font_weight, font_size
from cjm_fasthtml_tailwind.utilities.sizing import w
from cjm_fasthtml_tailwind.core.base import combine_classes, SingleValueUtility

## Pythonic SSE Configuration with Dataclasses

In [3]:
class SSEMessageType(Enum):
    """Types of SSE messages"""
    PROGRESS = "progress"
    ERROR = "error"
    COMPLETE = "complete"
    HEARTBEAT = "heartbeat"

@dataclass
class SSEConfig:
    """Configuration for SSE connection"""
    endpoint: str = "/stream"
    reconnect_delay: int = 3000  # milliseconds
    max_retries: int = 5
    heartbeat_timeout: int = 30000  # milliseconds
    
@dataclass
class ProgressUIConfig:
    """Configuration for progress UI elements"""
    progress_bar_selector: str = "#progress-display progress"
    status_text_selector: str = "#progress-display p:last-child"
    start_button_id: str = "start-btn"
    container_id: str = "progress-container"
    
    # Display formatting
    decimal_places: int = 1
    show_percentage: bool = True
    show_counts: bool = True
    
@dataclass
class SSEHandlers:
    """JavaScript event handlers for SSE"""
    on_progress: Optional[str] = None
    on_complete: Optional[str] = None
    on_error: Optional[str] = None
    custom_handlers: Dict[str, str] = field(default_factory=dict)

## SSE Manager Class - Pythonic JavaScript Generation

In [4]:
class SSEManager:
    """Generates JavaScript SSE management code from Python configuration"""
    
    def __init__(self, 
                 config: SSEConfig = None,
                 ui_config: ProgressUIConfig = None,
                 handlers: SSEHandlers = None):
        self.config = config or SSEConfig()
        self.ui_config = ui_config or ProgressUIConfig()
        self.handlers = handlers or SSEHandlers()
        
    def generate_manager_script(self) -> str:
        """Generate the complete SSE manager JavaScript"""
        return f"""
window.sseManager = {{
    config: {self._generate_config()},
    currentSource: null,
    retryCount: 0,
    
    {self._generate_connect_method()}
    
    {self._generate_message_handler()}
    
    {self._generate_error_handler()}
    
    {self._generate_disconnect_method()}
    
    {self._generate_ui_updaters()}
    
    {self._generate_custom_handlers()}
}};
"""
    
    def _generate_config(self) -> str:
        """Generate JavaScript config object"""
        return f"""{{
        endpoint: '{self.config.endpoint}',
        reconnectDelay: {self.config.reconnect_delay},
        maxRetries: {self.config.max_retries},
        heartbeatTimeout: {self.config.heartbeat_timeout},
        ui: {{
            progressBar: '{self.ui_config.progress_bar_selector}',
            statusText: '{self.ui_config.status_text_selector}',
            startButton: '{self.ui_config.start_button_id}',
            container: '{self.ui_config.container_id}',
            decimalPlaces: {self.ui_config.decimal_places},
            showPercentage: {'true' if self.ui_config.show_percentage else 'false'},
            showCounts: {'true' if self.ui_config.show_counts else 'false'}
        }}
    }}"""
    
    def _generate_connect_method(self) -> str:
        """Generate the connect method"""
        return f"""
    connect: function(jobId) {{
        // Close any existing connection
        if (this.currentSource) {{
            this.currentSource.close();
        }}
        
        // Reset retry count
        this.retryCount = 0;
        
        // Create new EventSource connection
        const url = this.config.endpoint + '?job_id=' + jobId;
        this.currentSource = new EventSource(url);
        
        // Set up message handler
        this.currentSource.onmessage = this.handleMessage.bind(this);
        
        // Set up error handler
        this.currentSource.onerror = this.handleError.bind(this);
        
        console.log('SSE connected to job:', jobId);
    }},"""
    
    def _generate_message_handler(self) -> str:
        """Generate the message handler"""
        custom_progress = self.handlers.on_progress or "// Custom progress handler not defined"
        custom_complete = self.handlers.on_complete or "// Custom complete handler not defined"
        
        return f"""
    handleMessage: function(event) {{
        const data = JSON.parse(event.data);
        
        // Update progress bar
        this.updateProgressBar(data.progress);
        
        // Update status text
        this.updateStatusText(data);
        
        // Custom progress handler
        {custom_progress}
        
        // Handle completion
        if (data.completed) {{
            this.handleCompletion();
            {custom_complete}
        }}
    }},"""
    
    def _generate_error_handler(self) -> str:
        """Generate the error handler"""
        custom_error = self.handlers.on_error or "// Custom error handler not defined"
        
        return f"""
    handleError: function(error) {{
        console.error('SSE Error:', error);
        
        // Custom error handler
        {custom_error}
        
        // Retry logic
        if (this.retryCount < this.config.maxRetries) {{
            this.retryCount++;
            console.log(`Retrying connection (attempt ${{this.retryCount}}/${{this.config.maxRetries}})...`);
            setTimeout(() => {{
                // Retry connection logic here if needed
            }}, this.config.reconnectDelay);
        }} else {{
            console.error('Max retries reached. Closing connection.');
            this.disconnect();
        }}
    }},"""
    
    def _generate_disconnect_method(self) -> str:
        """Generate the disconnect method"""
        return f"""
    disconnect: function() {{
        if (this.currentSource) {{
            this.currentSource.close();
            this.currentSource = null;
            console.log('SSE disconnected');
        }}
    }},"""
    
    def _generate_ui_updaters(self) -> str:
        """Generate UI update methods"""
        return f"""
    updateProgressBar: function(progress) {{
        const progressBar = document.querySelector(this.config.ui.progressBar);
        if (progressBar) {{
            progressBar.value = progress;
        }}
    }},
    
    updateStatusText: function(data) {{
        const statusText = document.querySelector(this.config.ui.statusText);
        if (!statusText) return;
        
        if (data.bars && Object.keys(data.bars).length > 0) {{
            const firstBar = Object.values(data.bars)[0];
            let text = firstBar.desc + ': ';
            
            if (this.config.ui.showPercentage) {{
                text += firstBar.pct.toFixed(this.config.ui.decimalPlaces) + '%';
            }}
            
            if (this.config.ui.showCounts) {{
                text += ` (${{firstBar.cur}}/${{firstBar.tot}})`;
            }}
            
            statusText.textContent = text;
        }} else {{
            let text = 'Processing...';
            if (this.config.ui.showPercentage) {{
                text += ` ${{data.progress.toFixed(this.config.ui.decimalPlaces)}}%`;
            }}
            statusText.textContent = text;
        }}
    }},
    
    handleCompletion: function() {{
        this.disconnect();
        
        // Update status text
        const statusText = document.querySelector(this.config.ui.statusText);
        if (statusText) {{
            statusText.textContent = 'Completed!';
        }}
        
        // Re-enable start button
        const startBtn = document.getElementById(this.config.ui.startButton);
        if (startBtn) {{
            startBtn.disabled = false;
            startBtn.classList.remove('{btn_behaviors.disabled}');
            startBtn.setAttribute('hx-post', '/start_task');
            if (typeof htmx !== 'undefined') {{
                htmx.process(startBtn);
            }}
        }}
    }},"""
    
    def _generate_custom_handlers(self) -> str:
        """Generate any custom handlers"""
        if not self.handlers.custom_handlers:
            return ""
        
        handlers = []
        for name, code in self.handlers.custom_handlers.items():
            handlers.append(f"""
    {name}: function() {{
        {code}
    }},""")
        
        return '\n'.join(handlers)
    
    def create_connection_script(self, job_id: str) -> Script:
        """Create a script tag that connects to SSE for a specific job"""
        return Script(f"window.sseManager.connect('{job_id}');")
    
    def create_disconnect_script(self) -> Script:
        """Create a script tag that disconnects SSE"""
        return Script("window.sseManager.disconnect();")

## Progress Component Factory

In [5]:
@dataclass
class ProgressComponentConfig:
    """Configuration for progress components"""
    max_value: int = 100
    initial_value: int = 0
    label_text: str = "Progress:"
    initial_status: str = "Waiting..."
    color_scheme: SingleValueUtility = progress_colors.primary
    show_label: bool = True
    custom_classes: Optional[str] = None

class ProgressComponentFactory:
    """Factory for creating progress UI components"""
    
    def __init__(self, config: ProgressComponentConfig = None):
        self.config = config or ProgressComponentConfig()
    
    def create_progress_bar(self, value: Optional[int] = None) -> Progress:
        """Create a styled progress bar"""
        value = value if value is not None else self.config.initial_value
        
        classes = combine_classes(
            progress,
            self.config.color_scheme,
            w.full,
            self.config.custom_classes
        )
        
        return Progress(
            value=str(value),
            max=str(self.config.max_value),
            cls=classes
        )
    
    def create_progress_label(self, text: Optional[str] = None) -> P:
        """Create a styled progress label"""
        text = text or self.config.label_text
        return P(text, cls=combine_classes(font_weight.bold))
    
    def create_status_message(self, text: Optional[str] = None) -> P:
        """Create a styled status message"""
        text = text or self.config.initial_status
        return P(text, cls=combine_classes(m.t(2), font_size.sm))
    
    def create_progress_display(self, 
                               value: Optional[int] = None,
                               status_text: Optional[str] = None) -> Div:
        """Create a complete progress display"""
        components = []
        
        if self.config.show_label:
            components.append(self.create_progress_label())
        
        components.extend([
            self.create_progress_bar(value),
            self.create_status_message(status_text)
        ])
        
        return Div(*components, id="progress-display")

In [6]:
@dataclass
class ButtonConfig:
    """Configuration for button components"""
    text: str = "Start Task"
    button_id: str = "start-btn"
    endpoint: str = "/start_task"
    target: str = "#progress-container"
    swap_mode: str = "innerHTML"
    color_scheme: SingleValueUtility = btn_colors.primary
    disabled: bool = False

class ButtonFactory:
    """Factory for creating button components"""
    
    def __init__(self, config: ButtonConfig = None):
        self.config = config or ButtonConfig()
    
    def create_start_button(self, disabled: Optional[bool] = None) -> Button:
        """Create a start button with appropriate state"""
        is_disabled = disabled if disabled is not None else self.config.disabled
        
        btn_classes = combine_classes(
            btn,
            self.config.color_scheme,
            btn_behaviors.disabled if is_disabled else ""
        )
        
        attrs = {
            "id": self.config.button_id,
            "cls": btn_classes,
            "disabled": is_disabled
        }
        
        if not is_disabled:
            attrs.update({
                "hx_post": self.config.endpoint,
                "hx_target": self.config.target,
                "hx_swap": self.config.swap_mode
            })
        
        return Button(self.config.text, **attrs)
    
    def create_disable_script(self) -> Script:
        """Create a script to disable the button"""
        return Script(f"""
            var btn = document.getElementById('{self.config.button_id}');
            if (btn) {{
                btn.disabled = true;
                btn.classList.add('{btn_behaviors.disabled}');
                btn.removeAttribute('hx-post');
            }}
        """)

## Application Setup

In [7]:
# Create app
app, rt = create_test_app(theme=DaisyUITheme.LIGHT)

# Initialize monitor and runner
monitor = ProgressMonitor()
runner = JobRunner(monitor)

# Configure SSE and UI components
sse_config = SSEConfig(
    endpoint="/stream",
    reconnect_delay=3000,
    max_retries=5
)

ui_config = ProgressUIConfig(
    decimal_places=1,
    show_percentage=True,
    show_counts=True
)

# Custom handlers (optional)
handlers = SSEHandlers(
    on_progress="console.log('Progress update:', data.progress);",
    on_complete="console.log('Task completed!');",
    on_error="console.error('Connection error:', error);"
)

# Create managers and factories
sse_manager = SSEManager(sse_config, ui_config, handlers)
progress_factory = ProgressComponentFactory(
    ProgressComponentConfig(
        label_text="Task Progress:",
        initial_status="Ready to start",
        color_scheme=progress_colors.primary
    )
)
button_factory = ButtonFactory(
    ButtonConfig(
        text="Start Processing",
        color_scheme=btn_colors.primary
    )
)

## Task Definition

In [8]:
# Define a simple task that uses tqdm
def simple_task():
    from tqdm import tqdm
    import time
    
    results = []
    for i in tqdm(range(100), desc="Processing items"):
        time.sleep(0.01)
        results.append(i * 2)
    return results

## Routes with Pythonic Components

In [9]:
@rt("/")
def index():
    return create_test_page(
        "Pythonic SSE Progress Demo",
        Div(
            H2("Pythonic Progress Tracking with SSE"),
            button_factory.create_start_button(disabled=False),
            Div(
                P("Ready", cls=combine_classes(font_weight.bold, m.t(4))),
                id="progress-container",
                cls=str(m.t(4))
            ),
            cls=str(p(8))
        ),
        # Add the dynamically generated SSE manager script
        Script(sse_manager.generate_manager_script())
    )

@rt("/start_task")
def start_task():
    job_id = str(uuid.uuid4())
    
    # Start the job
    runner.start(
        job_id,
        simple_task,
        patch_kwargs={
            "min_delta_pct": 5,
            "min_update_interval": 0.05,
            "emit_initial": True
        }
    )
    
    # Return initial UI with SSE connection
    return Div(
        # Disable button
        button_factory.create_disable_script(),
        # Progress display
        progress_factory.create_progress_display(
            value=0,
            status_text="Starting..."
        ),
        # Connect to SSE stream
        sse_manager.create_connection_script(job_id)
    )

@rt("/stream")
def stream(job_id: str):
    """SSE endpoint that streams progress updates"""
    return EventStream(
        sse_stream(
            monitor,
            job_id,
            interval=0.1,
            heartbeat=10.0,
            wait_for_start=True,
            start_timeout=5.0
        )
    )

## Start Server

In [10]:
# Start server for Jupyter
server = start_test_server(app)
display(HTMX())

In [11]:
# Stop server when done
server.stop()